In [4]:
!pip install langchain faiss-cpu openai transformers torch torchvision

  Using cached faiss_cpu-1.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (4.8 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvjitlink_cu12-

In [5]:
!pip install pdfplumber

To interact with PDFs (text, images, and tables) using open-source tools. This chatbot can answer questions based on the content of PDFs, including text and images, and tables.

1. **Extract Text and Images from PDF** using `pdfplumber` for text and `Pillow` for images.
2. **Store the Data**: Using `FAISS` to store text and image features.
3. **Use a Pre-trained Language Model (LLM)**: We will use OpenAI’s `gpt-3.5-turbo` (or an equivalent free model from HuggingFace) for answering questions.
4. **RAG Setup**: We will use `LangChain` to implement the RAG-based pipeline.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/Colab\ Notebooks/NLP/data/sample.pdf .

###**1. Extract Text, Images, and Tables from PDF**

In [6]:
import pdfplumber
from PIL import Image
import io

def extract_text_and_tables(pdf_path):
    text = ""
    tables = []
    images = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text()  # Extracting text from the page
            tables.extend(page.extract_tables())  # Extracting tables from the page
            for img in page.images:
                im = Image.open(io.BytesIO(img["stream"]))
                images.append(im)
    return text, tables, images

In [ ]:
# Test
pdf_path = "/content/sample.pdf"

text, tables, images = extract_text_and_tables(pdf_path)

print(text)
print(tables)
images[0].show()  # first extracted image

###**2. Storing Text & Image Embeddings and using FAISS for Efficient Similarity Search**

First, we’ll extract text embeddings using a pretrained language model and image embeddings using a vision model.

We can use `FAISS`(Facebook AI Similarity Search) to store text embeddings and image embeddings for efficient search. Faiss is a open-source library by facebook for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size.

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from transformers import CLIPProcessor, CLIPModel
import torch

# Get text embeddings
def get_text_embeddings(texts):
    model = HuggingFaceEmbeddings()
    return model.embed_documents(texts)

# Get image embeddings
def get_image_embeddings(images):
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
    image_inputs = processor(images=images, return_tensors="pt")
    with torch.no_grad():
        image_embeddings = model.get_image_features(**image_inputs)
    return image_embeddings

In [ ]:
# Test
images_embeddings = get_image_embeddings(images)
texts = [text]
texts_embeddings = get_text_embeddings(texts)

# Storing in FAISS
text_faiss = FAISS.from_documents(texts, HuggingFaceEmbeddings())
image_faiss = FAISS.from_vectors(images_embeddings, dimension=512)  # CLIP output is 512-d

# We can store both and search them separately based on the type of query.

###**3. Implementing RAG with LangChain**

Now, let’s integrate this with a RAG approach using LangChain. This will allow the chatbot to fetch relevant information from both text and images to answer a query.


- Convert the textual query into an image embedding using CLIP's text encoder.

- Perform a similarity search in the image_faiss index.

- Return the top image match(es) based on cosine similarity.

In [ ]:
from transformers import CLIPTokenizer, CLIPModel
import torch

# Load the CLIP model and tokenizer
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch16")

def get_text_embedding_for_image_query(query):
    inputs = clip_tokenizer([query], return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        text_features = clip_model.get_text_features(**inputs)
    return text_features

def search_similar_images(query, image_embeddings, top_k=1):
    # Convert query to embedding
    query_embedding = get_text_embedding_for_image_query(query)
    # Compute cosine similarity manually
    query_norm = torch.nn.functional.normalize(query_embedding, p=2, dim=1)
    image_norms = torch.nn.functional.normalize(image_embeddings, p=2, dim=1)
    cosine_sim = torch.mm(query_norm, image_norms.T).squeeze(0)
    # Get top matches
    top_indices = torch.topk(cosine_sim, top_k).indices.tolist()
    return top_indices

In [ ]:
from langchain.chains import RetrievalQA
from langchain.chains import RAG
from langchain.llms import OpenAI
from langchain.agents import initialize_agent
from langchain.agents import Tool
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)  # We can also use HuggingFace’s free models for this

# Use FAISS as a retriever for both text and image data
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=text_faiss.as_retriever())

def multimodal_rag(query):
    # Fetch relevant results for the query from text and images
    text_results = qa_chain.run(query)

    # Optionally added image search and retrieval logic. For example, using the image_faiss search to match query related to images
    # Image search based on query semantics
    top_img_indices = search_similar_images(query, image_embeddings, top_k=1)

    # Retrieve the top-matched image (or images)
    matched_images = [images[i] for i in top_img_indices]
    for img in matched_images:
        img.show()

    return text_results  # Returning text response

In [ ]:
# Test with a query
query = "What are the key insights in the PDF?"
response = multimodal_rag(query)
print(response)

In [ ]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader, ImageLoader
from langchain.embeddings import OpenAIEmbeddings
from PIL import Image
import os

# Load language model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

# if we already have these FAISS indexes built and saved:
text_faiss = FAISS.load_local("text_index", OpenAIEmbeddings())
image_faiss = FAISS.load_local("image_index", OpenAIEmbeddings())

# Build the QA chain for text retrieval
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=text_faiss.as_retriever()
)

# Define the multimodal RAG function
def multimodal_rag(query):
    # Retrieve text documents related to the query
    text_response = qa_chain.run(query)

    # Retrieve image metadata or captions from image index (assuming textual metadata was indexed)
    image_docs = image_faiss.similarity_search(query, k=3)
    image_results = "\n".join([doc.page_content for doc in image_docs])

    # Combine results
    combined_response = f"Text Results:\n{text_response}\n\nImage Results:\n{image_results}"
    return combined_response

# Example usage
query = "What are the safety features in electric vehicles?"
response = multimodal_rag(query)
print(response)


Note:

- text_faiss and image_faiss must be built using text content (text documents or image captions/metadata).

- If your image index stores actual images, you’d need to extract embeddings using a vision model (e.g., CLIP) and compare via cosine similarity.

- You can expand this with tools like LangChain Agents or LangGraph for more dynamic workflows.

###**4. Bringing It All Together**

In [ ]:
# Full process to extract data from PDF, process it, and query the chatbot.
pdf_path = "/content/sample.pdf"

text, tables = extract_text_and_tables(pdf_path)
images = extract_images(pdf_path)


texts = [text]  # List of texts to pass for embedding
texts_embeddings = get_text_embeddings(texts)
images_embeddings = get_image_embeddings(images)

# Store embeddings in FAISS
text_faiss = FAISS.from_documents(texts, HuggingFaceEmbeddings())
image_faiss = FAISS.from_vectors(images_embeddings, dimension=512)

# Define and run multimodal RAG-based chatbot
query = "What are the key takeaways in the PDF?"
response = multimodal_rag(query)

print(response)

This project setup extracts text, images, and tables from PDFs and uses `FAISS` for efficient data retrieval.

1. **Text Embedding**: We used HuggingFace's pre-trained transformer models.
2. **Image Embedding**: We used CLIP (a vision model from OpenAI) to get image features.
3. **RAG Model**: We used LangChain’s `RetrievalQA` to implement a retrieval-augmented generation system that helps answer questions based on the available text and image data.


In [ ]:
def multimodal_rag(query, images):
    # Step 1: Get relevant text answer
    text_response = qa_chain.run(query)

    # Step 2: Image search based on query semantics
    top_img_indices = search_similar_images(query, image_embeddings, top_k=1)

    # Retrieve the top-matched image (or images)
    matched_images = [images[i] for i in top_img_indices]

    # Optional: Display image (or return it as output)
    for img in matched_images:
        img.show()  # Or save/display elsewhere

    return text_response
